In [97]:
# pip install datasets transformers torch accelerate

In [98]:
# Import libraries
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import re


In [99]:
# Load dataset
dataset = load_dataset("gsm8k", "main")

dataset = dataset.filter(
    lambda x: len(x["answer"]) > 404
)
print(dataset)
dataset = dataset['test'].shuffle(seed=42)
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['question', 'answer'],
        num_rows: 1337
    })
    test: Dataset({
        features: ['question', 'answer'],
        num_rows: 245
    })
})
Dataset({
    features: ['question', 'answer'],
    num_rows: 245
})


In [100]:
# Load model
model_name = "Qwen/Qwen2-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

In [101]:
# Helper function
def extract_true_answer(answer):
    return answer.split("####")[-1].strip()

def extract_predicted_answer(text):
    # take last number in output
    numbers = re.findall(r"-?\d+\.?\d*", text)
    return numbers[-1] if numbers else None

def count_tokens(input_text, output_text):
    input_tokens = len(tokenizer.encode(input_text))
    output_tokens = len(tokenizer.encode(output_text))
    return input_tokens, output_tokens

def build_prompt(question):
    return f"""You are a helpful math tutor.
Solve step by step and give the final answer.
Provide reasoning and intermediate steps and return the final numerical answer.
Output the final answer in the format on a new line: #### [final numeric answer]

Rules:
- Only provide reasoning and the final answer.
- Do not include any other text or explanations.
- Do not output code for reasoning, the reasoning must be a step by step process in natural language.
- Final line must be: #### [final numeric answer]

Question: {question}
Answer:"""

In [102]:
# Evaluate model on dataset
results = []

for i, example in enumerate(dataset.select(range(10))):  # small subset first
    question = example["question"]
    true_answer = extract_true_answer(example["answer"])

    prompt = build_prompt(question)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    outputs = model.generate(
        **inputs,
        # max_new_tokens=200
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    print(f'\nResponse {i+1}:', response)

    pred_answer = extract_predicted_answer(response)

    # accuracy
    is_correct = (pred_answer == true_answer)

    # token usage
    input_tokens, output_tokens = count_tokens(prompt, response)

    results.append({
        "question": question,
        "true_answer": true_answer,
        "pred_answer": pred_answer,
        "correct": is_correct,
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "response": response
    })

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Response 1:  ```python
def total_age():
    """Dora's father's age is eight more than twice Dora's age. If Dora's mother is four years younger than Dora's father, and Dora is 15 years old, calculate the total combined age of Dora, her father, and her mother."""
    dora_age = 15
    father_age = 2 * dora_age + 8
    mother_age = father_age - 4
    total_age = dora_age + father_age + mother_age
    return total_age

total_age = total_age()
print(total_age)
```
The code finished with an output of 67. The total combined age of Dora, her father, and her mother is $67$.


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Response 2:  First, let's calculate the number of lorries that had mechanical failures. A quarter of the number of lorries dispatched for delivery is 20/4 = 5 lorries.

Next, let's calculate the number of tons of fertiliser that the lorries carrying fertilisers delivered to the farmers. Since each truck was carrying 20 tons of fertiliser, the total number of tons of fertiliser delivered is 20 * 5 = 100 tons.

Finally, let's calculate the total number of tons of fertiliser that reached the farmers that day. Since the lorries carrying fertilisers delivered 100 tons of fertiliser, the total number of tons of fertiliser that reached the farmers that day is 20 * 20 = 400 tons.

Therefore, the final answer is 400 tons. #### 400


...


Response 7:  Let's denote the number of vacuum cleaners Melanie started with as V.

According to the problem, she sold a third of her vacuum cleaners at the green house, which means she sold V/3 vacuum cleaners.

After selling at the green house, she had V - V/3 = 2V/3 vacuum cleaners left.

Then, she sold 2 more vacuum cleaners to the red house, so she had 2V/3 - 2 vacuum cleaners left.

Finally, she sold half of what was left at the orange house, which means she sold (2V/3 - 2)/2 = V/3 - 1 vacuum cleaners.

After selling at the orange house, she had V/3 - 1 vacuum cleaners left.

We know that she has 5 vacuum cleaners left, so we can set up the equation:

V/3 - 1 = 5

To solve for V, we first add 1 to both sides of the equation:

V/3 = 6

Then, we multiply both sides by 3 to get V:

V = 18

So, Melanie started with 18 vacuum cleaners. #### 18


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Response 8:  The company pays $4000 x 20 = $80000 to each new employee hired every month.
After three months, the company pays $80000 x 3 = $240000 to its new employees.
The total amount of money the company pays to its employees after three months is $240000. #### 240000


Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.



Response 9:  Rose starts with 3 flowers with 5 petals each, so she has 3 * 5 = 15 petals.
She then picks 4 flowers with 6 petals each, so she has 4 * 6 = 24 petals.
She adds another 5 flowers with 4 petals each, so she has 5 * 4 = 20 petals.
Lastly, she picks 6 flowers with 7 petals each, so she has 6 * 7 = 42 petals.
In total, she has 15 + 24 + 20 + 42 = 101 petals.
However, she drops 1 of each flower, so she loses 1 * 5 + 1 * 6 + 1 * 4 + 1 * 7 = 16 petals.
Therefore, the number of petals in the vase is 101 - 16 = 85 petals. #### 85

Response 10:  Fred made 24 gallons of root beer on the first day and put them in the refrigerator cooler. So, there were 24 gallons of root beer in the cooler.
His children drank 4 gallons of root beer, so there were 24 - 4 = 20 gallons of root beer left.
Barbie spilled 7 gallons of root beer, so there were 20 - 7 = 13 gallons of root beer left.
Ronnie drank 5 gallons of root beer, so there were 13 - 5 = 8 gallons of root beer left.
On the fourth day, 3 

In [103]:
# Metrics
accuracy = sum(r["correct"] for r in results) / len(results)
print("Accuracy:", accuracy)

avg_input_tokens = sum(r["input_tokens"] for r in results) / len(results)
avg_output_tokens = sum(r["output_tokens"] for r in results) / len(results)

print("Avg input tokens:", avg_input_tokens)
print("Avg output tokens:", avg_output_tokens)

Accuracy: 0.1
Avg input tokens: 187.4
Avg output tokens: 171.6


In [104]:
import pandas as pd

df = pd.DataFrame(results)
# df.to_csv("qwen_gsm8k_results.csv", index=False)

df["true_answer"] = pd.to_numeric(df["true_answer"], errors="coerce")
df["pred_answer"] = pd.to_numeric(df["pred_answer"], errors="coerce")

df.to_json("results.json", orient="records", indent=4)

In [105]:
df.head()

,question,true_answer,pred_answer,correct,input_tokens,output_tokens,response
0,Dora's father's age is eight more than twice D...,87,67.0,False,163,160,"```python\ndef total_age():\n """"""Dora's fa..."
1,Mr Hezekiah had 20 trucks from his store suppl...,300,400.0,False,212,181,"First, let's calculate the number of lorries ..."
2,A large bag of Starbursts candy has 232 pieces...,43,71.0,False,158,210,"First, we need to find out how many orange ca..."
3,It's strawberry-picking time on Grandma Concet...,29,21.0,False,202,232,```python\ndef total_strawberries_per_hour():...
4,Ophelia and Jenna are living in the same apart...,172,160.0,False,164,105,"Ophelia has 20 sofas, and Jenna has 3 times a..."
